In [1]:
import pandas as pd
import numpy as np
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from joblib import Parallel, delayed
from keybert import KeyBERT

In [2]:
# 1. 형태소 분석기 초기화
okt = Okt()

In [3]:
# 2. 사용자 정의 불용어 목록
stopwords = ['있다', '이', '그', '및', '등', '수', '을', '를', '에', '의', '가', '으로',
             '에서', '하는', '위해', '대한', '되다', '된다', '했다', '합니다', '및',
             '하는', '것', '그리고', '또한', '하여', '하고', '해서', '한다', '것이', '것을']


In [4]:
# 3. 명사 추출 + 필터링
def extract_nouns(text):
    nouns = okt.nouns(text)
    return [n for n in nouns if len(n) > 1 and n not in stopwords]

In [5]:
# 4. 뉴스 데이터 불러오기
df = pd.read_csv('../data/crawled_data.csv')

In [6]:
# 5. 병렬로 명사 추출
def extract_nouns(text):
    okt = Okt()
    nouns = okt.nouns(text)
    return [n for n in nouns if len(n) > 1 and n not in stopwords]

# 그대로 병렬 처리 가능
df['nouns'] = Parallel(n_jobs=-1)(delayed(extract_nouns)(text) for text in df['body'])

In [7]:
# 6. 명사 문자열로 변환
df['nouns_str'] = df['nouns'].apply(lambda x: ' '.join(x))


In [8]:
# 7. TF-IDF 벡터화 (단어 + 단어쌍)
tfidf = TfidfVectorizer(max_features=15000,
                        ngram_range=(1, 2),
                        stop_words=stopwords)
tfidf_matrix = tfidf.fit_transform(df['nouns_str'])

feature_names = np.array(tfidf.get_feature_names_out())


In [9]:
# 8. 각 뉴스에서 상위 키워드 추출

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2", device="cuda")
kw_model = KeyBERT(model)

def extract_top_keywords(text, top_n=5):
    keywords = kw_model.extract_keywords(
        text, top_n=top_n, stop_words=stopwords,
        use_mmr=True, diversity=0.7  # 중복 방지 설정
    )
    return [kw for kw, _ in keywords]
    
df['keywords'] = df['body'].apply(lambda x: extract_top_keywords(x, top_n=5))

In [10]:
# 9. 결과 출력
for i, row in df.iterrows():
    print(f"뉴스 제목: {row['title']}")
    print("키워드:")
    for kw in row['keywords']:
        print(f" - {kw}")
    print("="*50)

뉴스 제목: 허깅페이스, 맥북에서 구동하는 로봇용 시각언어행동 모델 출시
키워드:
 - 허깅페이스에서
 - vla
 - lerobot
 - metaworld
 - asynchronous
뉴스 제목: 레딧, 무단으로 AI 훈련 데이터 사용한 앤트로픽 고소
키워드:
 - 수십억달러에
 - 넘는
 - 7월
 - 강조했다
 - ai
뉴스 제목: 미스트랄, 코딩 AI 역량 총 집결한 기업용 어시스턴트 미스트랄 코드 출시
키워드:
 - 미스트랄은
 - devstral
 - git
 - jetbrains
 - code
뉴스 제목: 오픈AI, 업무용 AI 도구 대거 출시...MS구글과 에이전트 경쟁 불가피
키워드:
 - 마이크로소프트
 - pr
 - codex
 - canvas
 - connectors
뉴스 제목: 미중 갈등으로 애플 인텔리전스 중국 출시 지연
키워드:
 - 사이버스페이스
 - 가했다
 - 25
 - ai
 - cac
뉴스 제목: 퍼플렉시티, 에이닷 제치고 국내 모바일 AI 앱 3위 올라
키워드:
 - 와이즈앱리테일은
 - 결과
 - 됐다
 - mau
 - ios
뉴스 제목: AI 검색 광고 시장, 5년 내 26배로 증가...비중은 14에 그쳐
키워드:
 - 급성장하지만
 - seo
 - ai
 - geo
 - emarketer
뉴스 제목: 엔비디아 블랙웰, ML커먼스 테스트서 이전 세대보다 2배 빠른 성능 기록
키워드:
 - 엔비디아는
 - blackwell
 - mlperf
 - tyche
 - 405b
뉴스 제목: 앤드류 응 바이브 코딩은 필수...코딩 교육도 AI 잘 쓰기 위해 필수
키워드:
 - 스타트업에서도
 - ai가
 - 4대
 - 강조했다
 - ai
뉴스 제목: 앤트로픽, 오픈AI가 인수한 윈드서프에 클로드 접근 축소
키워드:
 - 비효율적이라는
 - devin
 - vibe
 - cursor
 - api
뉴스 제목: 앤트로픽, 오픈AI구글 상대 인재 영입률 10배 높아
키워드:
 - 시그널파이어는
 - ai
 - cto
 - linkedin

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



뉴스 제목: 최애 드라마나 영화, 이제 만화책으로 소장한다...중국홍콩 연구진 공동개발
키워드:
 - 알고리즘은
 - gan
 - curve
 - titanic
 - tcs
뉴스 제목: 카카오모빌리티 2억달러 투자 유치... 기업가치 3조원 돌파
키워드:
 - 카카오모빌리티는
 - carlyle
 - group
 - maasmobility
 - tpg
뉴스 제목: 미래학자 토마스 프레이 2030년 빅테크 기업, 교육 분야 주력할 것
키워드:
 - 동기저하요인을
 - frey
 - education
 - herring
 - ies
뉴스 제목: 팝스타 타린 서던 컴백... 알고 보니 AI 클론
키워드:
 - 오버레이한다라며
 - southern
 - hour
 - bts
 - pixel
뉴스 제목: 인터뷰 사회 도움되는 AI기술 연구할 것전해곤 지스트 교수팀, AI 기반 보행자 경로 예측 알고리즘 개발
키워드:
 - 모델링함으로써
 - 가능해졌다
 - 2021
 - ai
 - gcn
뉴스 제목: 박정현의 데이터사이언스 시작하기 ⑨ 알고리즘 3  클러스터링
키워드:
 - 데이터세트에는
 - clustering
 - kmeans
 - label
 - variable
뉴스 제목: 시중 판매 카메라로 정맥인식 기술 구현... 생체 인식 보안 문제 대안책
키워드:
 - university
 - wales
 - 복제에
 - d415
 - autoencoder
뉴스 제목: 일상복처럼 가볍고 편안한 의복형 웨어러블 로봇 출시 머지않았다
키워드:
 - 인공근육은
 - tech
 - xplore
 - 5g
 - bristol
뉴스 제목: 세심하고 전방위적인 취재 좋았으나, 깔끔하게 정리된 보도 아쉬워
키워드:
 - 동아사이언스
 - 21
 - ces2021
 - ai
 - jtbc
뉴스 제목: 동국대 LINC사업단, 지능형 소재부품장비 ICC 반년의 성과는
키워드:
 - 융합에너지신소재공학과
 - industrial
 - linc
 - allset
 - center
뉴스 제목: 네

In [11]:

# 10. 새 뉴스 키워드 추출 함수
def extract_keywords_from_new_news(news_text, vectorizer, stopwords, top_n=5):
    nouns = [n for n in okt.nouns(news_text) if len(n) > 1 and n not in stopwords]
    text = ' '.join(nouns)
    tfidf_vec = vectorizer.transform([text])
    row = tfidf_vec.toarray().flatten()
    top_indices = row.argsort()[-top_n:][::-1]
    feature_names = vectorizer.get_feature_names_out()
    keywords = [(feature_names[i], round(row[i], 3)) for i in top_indices if row[i] > 0]
    return keywords


In [12]:
# 11. 테스트
new_news = """ 제조 특화 인공지능(AI) 기업들이 현장 도입을 늘리기 위해 가격 경쟁력과 보안을 강조하고 나섰다. 11일부터 코엑스에서 열리는 ‘AI&빅데이터쇼’에는 제조 AI 기업들이 제품 시연과 집중적인 컨설팅을 진행했다. 

제조 분야에서는 ▲비전 AI를 활용한 제품의 불량 검출 ▲작업 현장 안전관제 솔루션 그리고 ▲기계 설비의 시계열 데이터를 활용한 이상 탐지 및 예지보전 솔루션 등이 주를 이뤘다. 

비전 AI 솔루션의 경우, 이미지 데이터 학습 과정에서 투자 비용을 줄이는 접근법을 보여줬다.

RTM은 AI 비전 검사 플랫폼 ‘허블’을 소개하며, 데이터 라벨링 과정에서 효율성을 높였다고 설명했다.

허블은 검사모델을 학습하고 배포하는 ‘엔진’과 설비에 연동하는 ‘엣지’, 그리고 데이터를 통합관리하는 '대시보드'로 이뤄져 있다. 이 중 엔진에 데이터를 학습할 때 제품 이미지에서 불량 부분을 인식하고 학습하는 과정을 자동화했다.

이로 인해 검사원 공수를 크게 83%까지 줄이고, 도입 비용도 합리적으로 제시할 수 있다는 설명이다."""
keywords = extract_keywords_from_new_news(new_news, tfidf, stopwords)
print("추출된 키워드:", keywords)

추출된 키워드: [('검사', np.float64(0.251)), ('불량', np.float64(0.242)), ('학습 과정', np.float64(0.207)), ('제조', np.float64(0.204)), ('설비', np.float64(0.186))]
